In [3]:
from google.colab import files

uploaded = files.upload()

Saving credit_risk_ml.csv to credit_risk_ml.csv


In [4]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path("/content/creditlens-risk-case-study")
DATA_PATH = PROJECT_ROOT / "data" / "processed"

DATA_PATH.mkdir(parents=True, exist_ok=True)

shutil.move(
    "/content/credit_risk_ml.csv",
    DATA_PATH / "credit_risk_ml.csv"
)

print("Project exists:", PROJECT_ROOT.exists())
print("Processed file:", DATA_PATH / "credit_risk_ml.csv")

Project exists: True
Processed file: /content/creditlens-risk-case-study/data/processed/credit_risk_ml.csv


In [5]:
import pandas as pd

MODEL_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "credit_risk_ml.csv"
)

data = pd.read_csv(MODEL_DATA_PATH)

print(f"Rows: {len(data):,}")
print(f"Columns: {len(data.columns)}")

Rows: 32,581
Columns: 19


In [6]:
NUMERIC_FEATURES = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
    "loan_term_months",
    "loan_to_income_ratio",
    "other_debt",
    "debt_to_income_ratio",
    "open_accounts",
    "credit_utilization_ratio",
    "past_delinquencies"
]

CATEGORICAL_FEATURES = [
    "loan_grade",
    "person_home_ownership",
    "loan_intent",
    "cb_person_default_on_file"
]

TARGET = "loan_status"

FEATURE_COLUMNS = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)

required_columns = FEATURE_COLUMNS + [TARGET]

missing_columns = [
    column
    for column in required_columns
    if column not in data.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Model inputs validated.")
print(f"Numeric features     : {len(NUMERIC_FEATURES)}")
print(f"Categorical features : {len(CATEGORICAL_FEATURES)}")
print(f"Total predictors     : {len(FEATURE_COLUMNS)}")
print(f"Target               : {TARGET}")

Model inputs validated.
Numeric features     : 14
Categorical features : 4
Total predictors     : 18
Target               : loan_status


In [7]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES
        )
    ]
)

model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            model
        )
    ]
)

print("Modelling pipeline created.")
print("Preprocessing: imputation + scaling + one-hot encoding")
print("Classifier: Logistic Regression")
print("Class weighting: balanced")

Modelling pipeline created.
Preprocessing: imputation + scaling + one-hot encoding
Classifier: Logistic Regression
Class weighting: balanced


In [8]:
X = data[FEATURE_COLUMNS]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training observations:", f"{len(X_train):,}")
print("Test observations    :", f"{len(X_test):,}")

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTest target distribution:")
print(y_test.value_counts().sort_index())

Training observations: 26,064
Test observations    : 6,517

Training target distribution:
loan_status
0    20378
1     5686
Name: count, dtype: int64

Test target distribution:
loan_status
0    5095
1    1422
Name: count, dtype: int64


In [9]:
pipeline.fit(
    X_train,
    y_train
)

print("Model training completed.")

Model training completed.


In [10]:
test_probabilities = pipeline.predict_proba(
    X_test
)[:, 1]

test_predictions = pipeline.predict(
    X_test
)

roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)

pr_auc = average_precision_score(
    y_test,
    test_probabilities
)

print("Model performance")
print("-" * 40)
print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")

print("\nClassification report")
print("-" * 40)

print(
    classification_report(
        y_test,
        test_predictions,
        digits=4
    )
)

Model performance
----------------------------------------
ROC-AUC : 0.8784
PR-AUC  : 0.7389

Classification report
----------------------------------------
              precision    recall  f1-score   support

           0     0.9329    0.8290    0.8779      5095
           1     0.5621    0.7862    0.6555      1422

    accuracy                         0.8197      6517
   macro avg     0.7475    0.8076    0.7667      6517
weighted avg     0.8520    0.8197    0.8294      6517



In [12]:
all_probabilities = pipeline.predict_proba(
    X
)[:, 1]

scored_data = X.copy()

scored_data["loan_status"] = y.values

scored_data["pd_score"] = (
    all_probabilities
    .clip(0, 1)
    .round(6)
)

print("PD scoring completed.")
print(f"Rows scored: {len(scored_data):,}")

print("\nPD score summary")
print("-" * 40)
print(scored_data["pd_score"].describe())

PD scoring completed.
Rows scored: 32,581

PD score summary
----------------------------------------
count    32581.000000
mean         0.377027
std          0.296617
min          0.000616
25%          0.126576
50%          0.281524
75%          0.596819
max          0.999949
Name: pd_score, dtype: float64


In [14]:
scored_data = X.copy()

scored_data["loan_status"] = y.values

scored_data["pd_score"] = (
    pipeline.predict_proba(X)[:, 1]
    .clip(0, 1)
    .round(6)
)

output_path = (
    "/content/creditlens-risk-case-study/"
    "data/processed/credit_risk_scored.csv"
)

scored_data.to_csv(
    output_path,
    index=False
)

print("Scored dataset created.")
print(f"Rows   : {len(scored_data):,}")
print(f"Columns: {len(scored_data.columns)}")
print(f"Output : {output_path}")

print("\nSample")
print(scored_data.head())

Scored dataset created.
Rows   : 32,581
Columns: 20
Output : /content/creditlens-risk-case-study/data/processed/credit_risk_scored.csv

Sample
   person_age  person_income  person_emp_length  loan_amnt  loan_int_rate  \
0          22          59000              123.0      35000          16.02   
1          21           9600                5.0       1000          11.14   
2          25           9600                1.0       5500          12.87   
3          23          65500                4.0      35000          15.23   
4          24          54400                8.0      35000          14.27   

   loan_percent_income  cb_person_cred_hist_length  loan_term_months  \
0                 0.59                           3                36   
1                 0.10                           2                36   
2                 0.57                           3                36   
3                 0.53                           2                12   
4                 0.55            

In [15]:
import joblib
import os

model_path = (
    "/content/creditlens-risk-case-study/"
    "models/logistic_model.pkl"
)

os.makedirs(
    os.path.dirname(model_path),
    exist_ok=True
)

joblib.dump(
    pipeline,
    model_path
)

print("Model pipeline serialized successfully.")
print(f"Output: {model_path}")

Model pipeline serialized successfully.
Output: /content/creditlens-risk-case-study/models/logistic_model.pkl


In [17]:
model_path = (
    "/content/creditlens-risk-case-study/"
    "models/logistic_model.pkl"
)

scored_path = (
    "/content/creditlens-risk-case-study/"
    "data/processed/credit_risk_scored.csv"
)

model_features = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
    "loan_term_months",
    "loan_to_income_ratio",
    "other_debt",
    "debt_to_income_ratio",
    "open_accounts",
    "credit_utilization_ratio",
    "past_delinquencies",
    "loan_grade",
    "person_home_ownership",
    "loan_intent",
    "cb_person_default_on_file"
]

# Reload serialized pipeline
loaded_pipeline = joblib.load(model_path)

# Reload scored dataset
saved_scores = pd.read_csv(scored_path)

# Recalculate PD for a small sample
sample_X = saved_scores[model_features].head(10)

reloaded_probabilities = (
    loaded_pipeline
    .predict_proba(sample_X)[:, 1]
    .clip(0, 1)
)

saved_probabilities = (
    saved_scores["pd_score"]
    .head(10)
    .to_numpy()
)

max_difference = abs(
    reloaded_probabilities - saved_probabilities
).max()

print("CreditLens model QA")
print("-" * 40)
print(f"Model file exists   : {os.path.exists(model_path)}")
print(f"Scored file exists  : {os.path.exists(scored_path)}")
print(f"Scored rows         : {len(saved_scores):,}")
print(f"PD column present   : {'pd_score' in saved_scores.columns}")
print(f"Max score difference: {max_difference:.10f}")

CreditLens model QA
----------------------------------------
Model file exists   : True
Scored file exists  : True
Scored rows         : 32,581
PD column present   : True
Max score difference: 0.0000004696
